<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/Optuna/z347_Optuna_LightGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grupo 3: Optuna + LightGBM

## Banegas - Marín - Mengoni - Rey

Recibe el dataset de Feature Engineering ya preprocesado y:
1. Detecta y filtra variables con posible data leakage
2. Corre Optuna para optimizar LightGBM
3. Visualiza la evolución de trials y parámetros
4. Entrena el modelo final con los mejores hiperparámetros

**Apartado al final**: warm-starting para clusters — cómo reutilizar lo aprendido en la búsqueda global cuando arrancás una búsqueda por cluster.

In [ ]:
!pip install uv -q
!uv pip install -q lightgbm optuna optuna-integration[lightgbm] plotly duckdb google-cloud-storage

In [ ]:
import os, warnings, json, tempfile
import numpy as np
import duckdb
import lightgbm as lgb
import optuna
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from google.cloud import storage as gcs

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

con = duckdb.connect()   # conexión en memoria

# ── PARAM ────────────────────────────────────────────────────────────────────
PARAM = {
    # Ruta al dataset de FE preprocesado (recibido del grupo de FE)
    # puede ser local o gs://bucket/path/archivo.csv
    'dataset_path':      'gs://natalialabo3_bukito2026/fe/dataset_fe.csv',

    # Columna target
    'target':            'clase',

    # Columnas que NO son features (id, periodo, target)
    'cols_excluir':      ['product_id', 'periodo', 'clase'],

    # Umbral de correlación con el target para alertar data leakage
    'umbral_leakage':    0.98,

    # Umbral de correlación entre features (multicolinealidad extrema)
    'umbral_multicol':   0.995,

    # Optuna
    'n_trials':          80,
    'n_folds':           5,
    'semilla':           102191,
    'n_jobs_lgbm':       -1,

    # Google Cloud Storage — bucket donde se guardan los resultados
    'gcs_bucket':        'natalialabo3_bukito2026',
    'gcs_prefix':        'optuna',           # carpeta dentro del bucket
}

# helper para subir archivos locales al bucket
def subir_a_gcs(ruta_local, nombre_blob):
    client = gcs.Client()
    bucket = client.bucket(PARAM['gcs_bucket'])
    blob   = bucket.blob(f"{PARAM['gcs_prefix']}/{nombre_blob}")
    blob.upload_from_filename(ruta_local)
    uri = f"gs://{PARAM['gcs_bucket']}/{PARAM['gcs_prefix']}/{nombre_blob}"
    print(f'  subido → {uri}')
    return uri

print('OK')

# 1. Carga del dataset de Feature Engineering

Este dataset todavía no lo tenemos — deberíamos esperar a que nos lo pase el grupo 2 y ahí leerlo desde el path en GCS.

In [ ]:
# DuckDB lee directo desde GCS sin pasar por pandas
# (la VM en GCP usa las credenciales del service account automáticamente)
con.execute(f"""
    INSTALL httpfs;
    LOAD httpfs;
    CREATE OR REPLACE TABLE dataset AS
    SELECT * FROM read_csv_auto('{PARAM['dataset_path']}', header=true)
""")

n_filas = con.execute("SELECT COUNT(*) FROM dataset").fetchone()[0]
cols    = [r[0] for r in con.execute("DESCRIBE dataset").fetchall()]
n_cols  = len(cols)

print(f'Dataset: {n_filas:,} filas × {n_cols} columnas')
print(f'Columnas (primeras 10): {cols[:10]}')
print()
con.execute("SELECT * FROM dataset LIMIT 3").df()

In [ ]:
cols_features = [c for c in cols if c not in PARAM['cols_excluir']]
print(f'Features disponibles: {len(cols_features)}')

stats_target = con.execute(f"""
    SELECT
        AVG({PARAM['target']})    AS media,
        STDDEV({PARAM['target']}) AS std,
        MIN({PARAM['target']})    AS minimo,
        MAX({PARAM['target']})    AS maximo
    FROM dataset
""").fetchone()

print(f'y — media: {stats_target[0]:.4f}  std: {stats_target[1]:.4f}  '
      f'min: {stats_target[2]:.4f}  max: {stats_target[3]:.4f}')

# 2. Detección de data leakage

Una feature tiene data leakage si:
- **Correlación con el target > umbral**: la feature "conoce" el futuro directamente
- **Nombre sospechoso**: contiene `t+`, `lag_-`, `_futuro`, `t-1`, `t-2`, `forward`, etc.
- **Multicolinealidad extrema**: dos features casi idénticas

In [ ]:
print('── Análisis de data leakage ──')
print()

# 1. Correlación con el target
corr_exprs = ", ".join(
    f"ABS(CORR({c}, {PARAM['target']})) AS {c}"
    for c in cols_features
)
corr_row  = con.execute(f"SELECT {corr_exprs} FROM dataset").fetchone()
corr_dict = dict(zip(cols_features, corr_row))
corr_sorted = sorted(corr_dict.items(), key=lambda x: -(x[1] or 0))

sospechosas_corr = [c for c, r in corr_sorted if (r or 0) > PARAM['umbral_leakage']]

print(f'1. Correlación con target > {PARAM["umbral_leakage"]}:')
if sospechosas_corr:
    for c in sospechosas_corr:
        print(f'   ⚠  {c:40s}  r={corr_dict[c]:.4f}')
else:
    print('   Ninguna — OK')

# 2. Nombres sospechosos
patrones_leakage = ['_-', 'lag_-', 'futuro', 't-1', 't-2', 't+', 'forward']
sospechosas_nombre = [c for c in cols_features
                      if any(p in c.lower() for p in patrones_leakage)]
print(f'\n2. Nombres sospechosos (lags negativos / forward):')
if sospechosas_nombre:
    for c in sospechosas_nombre:
        print(f'   ⚠  {c}')
else:
    print('   Ninguna — OK')

# 3. Top 20
print(f'\n3. Top 20 features por correlación con target:')
for c, r in corr_sorted[:20]:
    print(f'   {c:45s}  {(r or 0):.4f}')

cols_leakage = list(set(sospechosas_corr + sospechosas_nombre))
print(f'\nTotal features con flag de leakage: {len(cols_leakage)}')
if cols_leakage:
    print('  → Serán excluidas. Revisar manualmente antes de confirmar.')
    print(' ', cols_leakage)

In [ ]:
# Multicolinealidad extrema
print('── Multicolinealidad extrema ──')

pares_multicol = []
n_feat = len(cols_features)

for i in range(n_feat):
    for j in range(i + 1, n_feat):
        c1, c2 = cols_features[i], cols_features[j]
        r = con.execute(
            f"SELECT ABS(CORR({c1}, {c2})) FROM dataset"
        ).fetchone()[0] or 0
        if r > PARAM['umbral_multicol']:
            pares_multicol.append((c1, c2, r))

if pares_multicol:
    print(f'Pares con correlación > {PARAM["umbral_multicol"]}:')
    for c1, c2, r in sorted(pares_multicol, key=lambda x: -x[2]):
        print(f'  {c1:35s} ↔ {c2:35s}  r={r:.4f}')
else:
    print('Ningún par con multicolinealidad extrema — OK')

In [ ]:
# Decidir qué columnas usar — agregar a COLS_EXCLUIR_MANUAL si querés forzar exclusión
COLS_EXCLUIR_MANUAL = []

cols_leakage_final = list(set(cols_leakage + COLS_EXCLUIR_MANUAL))
cols_train = [c for c in cols_features if c not in cols_leakage_final]

print(f'Features originales:   {len(cols_features)}')
print(f'Excluidas por leakage: {len(cols_leakage_final)}')
print(f'Features para train:   {len(cols_train)}')

# única conversión a numpy — LightGBM/sklearn la necesitan
cols_sql = ", ".join(cols_train)
X = con.execute(f"SELECT {cols_sql} FROM dataset").fetchnumpy()
X = np.column_stack([X[c] for c in cols_train]).astype(np.float32)
y = con.execute(f"SELECT {PARAM['target']} FROM dataset").fetchnumpy()[PARAM['target']].astype(np.float32)

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

# 3. Optuna — búsqueda de hiperparámetros LightGBM

In [ ]:
def objective(trial):
    params = {
        'objective':         'regression',
        'metric':            'rmse',
        'verbosity':         -1,
        'boosting_type':     'gbdt',
        'n_jobs':            PARAM['n_jobs_lgbm'],
        'random_state':      PARAM['semilla'],

        'n_estimators':      trial.suggest_int('n_estimators', 100, 2000),
        'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 16, 256),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample':         trial.suggest_float('subsample', 0.4, 1.0),
        'subsample_freq':    1,
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_split_gain':    trial.suggest_float('min_split_gain', 0.0, 1.0),
    }

    kf = KFold(n_splits=PARAM['n_folds'], shuffle=True, random_state=PARAM['semilla'])
    rmses = []

    for fold, (idx_tr, idx_val) in enumerate(kf.split(X)):
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X[idx_tr], y[idx_tr],
            eval_set=[(X[idx_val], y[idx_val])],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(-1)],
        )
        pred = model.predict(X[idx_val])
        rmses.append(mean_squared_error(y[idx_val], pred, squared=False))

        trial.report(float(np.mean(rmses)), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(rmses))


print('Función objetivo definida.')

In [ ]:
sampler = optuna.samplers.TPESampler(seed=PARAM['semilla'])
pruner  = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2)

study = optuna.create_study(
    direction='minimize',
    sampler=sampler,
    pruner=pruner,
    study_name='lgbm_global',
)

print(f'Corriendo {PARAM["n_trials"]} trials...')
study.optimize(objective, n_trials=PARAM['n_trials'], show_progress_bar=True)

print(f'\nMejor RMSE: {study.best_value:.4f}')
print('Mejores hiperparámetros:')
for k, v in study.best_params.items():
    print(f'  {k:20s}: {v}')

# 4. Visualización — evolución de trials y parámetros

In [ ]:
trials_data = [
    {
        'number': t.number,
        'value':  t.value,
        'state':  t.state.name,
        **t.params,
    }
    for t in study.trials
]

con.execute("CREATE OR REPLACE TABLE trials AS SELECT * FROM trials_data")

trials_ok = con.execute("""
    SELECT *,
           MIN(value) OVER (ORDER BY number ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
               AS best_so_far
    FROM trials
    WHERE state = 'COMPLETE'
    ORDER BY number
""").df()

print(f'Trials completados: {len(trials_ok)}')
print(f'Trials podados:     {len(trials_data) - len(trials_ok)}')

### Evolución RMSE por trial

Puntos azules = un trial. Línea roja = mejor RMSE acumulado (solo baja).

**¿Qué esperar?** Los primeros ~10 trials son casi aleatorios (TPE todavía no aprendió). A partir del trial ~15-20 los puntos se agrupan en la zona baja. Si la línea roja sigue bajando al trial 80 → necesitás más trials. Si los puntos nunca convergen → el espacio de búsqueda es muy grande o el dataset tiene mucho ruido.

In [ ]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['RMSE por trial', 'Distribución de RMSE'])

fig.add_trace(go.Scatter(
    x=trials_ok['number'], y=trials_ok['value'],
    mode='markers', marker=dict(size=4, color='steelblue', opacity=0.6),
    name='RMSE trial',
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=trials_ok['number'], y=trials_ok['best_so_far'],
    mode='lines', line=dict(color='tomato', width=2),
    name='mejor hasta ahora',
), row=1, col=1)

fig.add_trace(go.Histogram(
    x=trials_ok['value'], nbinsx=30,
    marker_color='steelblue', opacity=0.7, name='distribución',
), row=1, col=2)
fig.add_vline(x=study.best_value, line_dash='dash', line_color='tomato',
              annotation_text=f'mejor={study.best_value:.4f}', row=1, col=2)

fig.update_layout(title='Evolución de trials Optuna', height=400)
fig.show()

### Importancia de parámetros (fANOVA)

**¿Qué esperar?** `learning_rate` y `num_leaves` suelen dominar. Parámetros con importancia baja (reg_alpha, min_split_gain) podés fijarlos y ahorrar trials en la próxima corrida.

In [ ]:
importances = optuna.importance.get_param_importances(study)

fig = go.Figure(go.Bar(
    x=list(importances.values()),
    y=list(importances.keys()),
    orientation='h',
    marker_color='steelblue',
))
fig.update_layout(
    title='Importancia de hiperparámetros (fANOVA)',
    xaxis_title='importancia relativa',
    height=400,
    yaxis=dict(autorange='reversed'),
)
fig.show()

### Evolución de cada hiperparámetro

Un subplot por parámetro. Eje X = trial, eje Y = valor del parámetro. Verde = RMSE bajo, rojo = RMSE alto.

**¿Qué esperar?** Los puntos verdes se concentran en una zona → ahí está el rango óptimo. Si los verdes están en los extremos del rango definido → el rango está cortando el óptimo, hay que expandirlo.

In [ ]:
param_cols = [c for c in trials_ok.columns
              if c not in ('number', 'value', 'state', 'best_so_far')]
ncols = 3
nrows = (len(param_cols) + ncols - 1) // ncols

fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=param_cols)

for i, col in enumerate(param_cols):
    r, c = divmod(i, ncols)
    fig.add_trace(go.Scatter(
        x=trials_ok['number'],
        y=trials_ok[col],
        mode='markers',
        marker=dict(
            size=5,
            color=trials_ok['value'],
            colorscale='RdYlGn_r',
            showscale=(i == 0),
            colorbar=dict(title='RMSE') if i == 0 else None,
        ),
        showlegend=False,
    ), row=r+1, col=c+1)

fig.update_layout(
    title='Evolución de hiperparámetros (verde=RMSE bajo, rojo=alto)',
    height=200 * nrows,
)
fig.show()

In [ ]:
# ── Parallel coordinates — top 30 trials ──
top30 = con.execute("""
    SELECT * FROM trials
    WHERE state = 'COMPLETE'
    ORDER BY value ASC
    LIMIT 30
""").df()

dims = [dict(label='RMSE', values=top30['value'])]
for col in param_cols:
    if col in top30.columns and top30[col].nunique() > 1:
        dims.append(dict(label=col, values=top30[col]))

fig = go.Figure(go.Parcoords(
    line=dict(color=top30['value'], colorscale='RdYlGn_r',
              showscale=True, colorbar=dict(title='RMSE')),
    dimensions=dims,
))
fig.update_layout(
    title='Parallel coordinates — top 30 trials (arrastrá los ejes para filtrar)',
    height=450,
)
fig.show()

# 5. Modelo final — entrenamiento con mejores hiperparámetros

In [ ]:
best_params = {
    **study.best_params,
    'objective':      'regression',
    'metric':         'rmse',
    'verbosity':      -1,
    'boosting_type':  'gbdt',
    'n_jobs':         PARAM['n_jobs_lgbm'],
    'random_state':   PARAM['semilla'],
    'subsample_freq': 1,
}

kf = KFold(n_splits=PARAM['n_folds'], shuffle=True, random_state=PARAM['semilla'])
rmses_final  = []
modelos_fold = []
splits_idx   = list(kf.split(X))

for fold, (idx_tr, idx_val) in enumerate(splits_idx):
    m = lgb.LGBMRegressor(**best_params)
    m.fit(X[idx_tr], y[idx_tr],
          eval_set=[(X[idx_val], y[idx_val])],
          callbacks=[lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(-1)])
    pred = m.predict(X[idx_val])
    rmse = mean_squared_error(y[idx_val], pred, squared=False)
    rmses_final.append(rmse)
    modelos_fold.append(m)
    print(f'  Fold {fold+1}: RMSE = {rmse:.4f}')

print(f'\nRMSE final CV: {np.mean(rmses_final):.4f} ± {np.std(rmses_final):.4f}')

In [ ]:
importances_lgbm = np.mean(
    [m.feature_importances_ for m in modelos_fold], axis=0
)

fi_data = [{'feature': f, 'importance': float(imp)}
           for f, imp in zip(cols_train, importances_lgbm)]
con.execute("CREATE OR REPLACE TABLE feature_importance AS SELECT * FROM fi_data")

fi_top = con.execute("""
    SELECT feature, importance
    FROM feature_importance
    ORDER BY importance DESC
    LIMIT 30
""").df()

fig = go.Figure(go.Bar(
    x=fi_top['importance'],
    y=fi_top['feature'],
    orientation='h',
    marker_color='steelblue',
))
fig.update_layout(
    title='Feature importance LightGBM — top 30 (promedio entre folds)',
    xaxis_title='importance (gain)',
    height=max(400, len(fi_top) * 18),
    yaxis=dict(autorange='reversed'),
)
fig.show()

In [ ]:
idx_tr0, idx_val0 = splits_idx[0]
pred_val = modelos_fold[0].predict(X[idx_val0])
y_val0   = y[idx_val0]
residuos = pred_val - y_val0

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Predicho vs real (fold 1)', 'Distribución de residuos'])

fig.add_trace(go.Scatter(
    x=y_val0.tolist(), y=pred_val.tolist(), mode='markers',
    marker=dict(size=3, opacity=0.4, color='steelblue'),
    name='pred vs real',
), row=1, col=1)
lim = float(max(y_val0.max(), pred_val.max()))
fig.add_trace(go.Scatter(x=[0, lim], y=[0, lim], mode='lines',
    line=dict(dash='dash', color='gray'), name='perfecto'), row=1, col=1)

fig.add_trace(go.Histogram(
    x=residuos.tolist(), nbinsx=50,
    marker_color='steelblue', opacity=0.7, name='residuos',
), row=1, col=2)
fig.add_vline(x=0, line_dash='dash', line_color='tomato', row=1, col=2)

fig.update_layout(title='Calidad del modelo — fold 1', height=400)
fig.show()

# 6. Guardar resultados en Google Cloud Storage

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:

    # best_params.json
    ruta_params = os.path.join(tmpdir, 'best_params_global.json')
    with open(ruta_params, 'w') as f:
        json.dump(study.best_params, f, indent=2)
    subir_a_gcs(ruta_params, 'best_params_global.json')

    # trials.parquet
    ruta_trials = os.path.join(tmpdir, 'trials_global.parquet')
    con.execute(f"""
        COPY (
            SELECT * FROM trials WHERE state = 'COMPLETE' ORDER BY number
        ) TO '{ruta_trials}' (FORMAT PARQUET)
    """)
    subir_a_gcs(ruta_trials, 'trials_global.parquet')

    # feature_importance.parquet
    ruta_fi = os.path.join(tmpdir, 'feature_importance.parquet')
    con.execute(f"""
        COPY (
            SELECT * FROM feature_importance ORDER BY importance DESC
        ) TO '{ruta_fi}' (FORMAT PARQUET)
    """)
    subir_a_gcs(ruta_fi, 'feature_importance.parquet')

print()
print('Resumen final:')
print(f'  RMSE CV:         {np.mean(rmses_final):.4f} ± {np.std(rmses_final):.4f}')
print(f'  Trials OK:       {len(trials_ok)}')
print(f'  Trials podados:  {len(trials_data) - len(trials_ok)}')
print(f'  Features usadas: {len(cols_train)}')
print(f'  Bucket:          gs://{PARAM["gcs_bucket"]}/{PARAM["gcs_prefix"]}/')

---
# APARTADO — Warm-starting para clusters

**La hipótesis**: si separo los productos en clusters y quiero optimizar un LightGBM para cada cluster, ¿tengo que arrancar Optuna desde cero o puedo reutilizar lo que aprendió en la búsqueda global?

**Lo que creemos**: tal vez se puede reutilizar. Optuna guarda cada trial como un objeto con parámetros + resultado. Podés "sembrar" un nuevo estudio con los mejores trials del global — el sampler TPE arranca con esa información en lugar de explorar al azar.

**¿Cuándo conviene?**
- Cluster con pocos productos → poco dato → warm-start ahorra trials
- Cluster muy distinto al promedio global → el warm-start puede confundir → la fase libre lo corregirá

In [ ]:
def crear_estudio_cluster(study_global, cluster_id,
                          X_cluster, y_cluster,
                          n_trials_warm=20, n_trials_new=40,
                          top_k_trials=10, semilla=102191):
    """
    Crea un estudio Optuna para un cluster específico,
    sembrado con los mejores trials del estudio global.

    1. Tomar top_k_trials mejores del estudio global
    2. Enqueue esos parámetros en el nuevo estudio
    3. Fase warm: evaluar esos parámetros en el cluster
    4. Fase libre: TPE explora desde lo aprendido
    """
    mejores_trials = sorted(
        [t for t in study_global.trials
         if t.state == optuna.trial.TrialState.COMPLETE],
        key=lambda t: t.value
    )[:top_k_trials]

    study_cluster = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=semilla + cluster_id),
        study_name=f'lgbm_cluster_{cluster_id}',
    )

    for trial in mejores_trials:
        study_cluster.enqueue_trial(trial.params)

    def objective_cluster(trial):
        params = {
            'objective':         'regression',
            'metric':            'rmse',
            'verbosity':         -1,
            'n_jobs':            -1,
            'random_state':      semilla,
            'subsample_freq':    1,
            'n_estimators':      trial.suggest_int('n_estimators', 100, 2000),
            'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'num_leaves':        trial.suggest_int('num_leaves', 16, 256),
            'max_depth':         trial.suggest_int('max_depth', 3, 12),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'subsample':         trial.suggest_float('subsample', 0.4, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 1.0),
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'min_split_gain':    trial.suggest_float('min_split_gain', 0.0, 1.0),
        }
        kf = KFold(n_splits=3, shuffle=True, random_state=semilla)
        rmses = []
        for idx_tr, idx_val in kf.split(X_cluster):
            m = lgb.LGBMRegressor(**params)
            m.fit(X_cluster[idx_tr], y_cluster[idx_tr],
                  eval_set=[(X_cluster[idx_val], y_cluster[idx_val])],
                  callbacks=[lgb.early_stopping(30, verbose=False),
                             lgb.log_evaluation(-1)])
            pred = m.predict(X_cluster[idx_val])
            rmses.append(mean_squared_error(y_cluster[idx_val], pred, squared=False))
        return float(np.mean(rmses))

    study_cluster.optimize(objective_cluster,
                           n_trials=min(top_k_trials, n_trials_warm),
                           show_progress_bar=False)
    study_cluster.optimize(objective_cluster,
                           n_trials=n_trials_new,
                           show_progress_bar=False)

    return study_cluster


print('Función warm-start definida.')
print()
print('Uso:')
print('  study_cl0 = crear_estudio_cluster(study, cluster_id=0,')
print('                                    X_cluster=X_cl0, y_cluster=y_cl0)')
print('  print(study_cl0.best_value, study_cl0.best_params)')

In [ ]:
print("""
  ESTUDIO GLOBAL (todos los productos)
  ─────────────────────────────────────
  80 trials → TPE aprende el landscape de hiperparámetros
  best_params: { n_estimators: 800, lr: 0.05, ... }
         │
         │  top_k_trials (ej: 10 mejores)
         │  se copian como seed al nuevo estudio
         ▼
  ESTUDIO CLUSTER 0 (ej: Mayonesas)
  ──────────────────────────────────
  Fase warm  (10 trials): evalúa los parámetros globales en el cluster
  Fase libre (40 trials): TPE explora desde lo que aprendió
  → converge más rápido que arrancar desde cero

  ESTUDIO CLUSTER 1 (ej: Jabones)
  ──────────────────────────────────
  Idem — misma semilla del global, distinto cluster_id

  Ventaja: si Mayonesas y Jabones tienen comportamiento similar,
  los parámetros globales ya son un buen punto de partida.
  Si son muy distintos, la fase libre lo corregirá.
""")

In [ ]:
# Demo comparativa cold-start vs warm-start
# Poner en True para correr. Requiere que ya exista 'study' del paso 3.
RUN_WARMSTART_DEMO = False

if RUN_WARMSTART_DEMO:
    rng  = np.random.default_rng(42)
    mask = rng.random(len(X)) < 0.5
    X_demo = X[mask]
    y_demo = y[mask]

    study_cold = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=0),
    )
    study_cold.optimize(lambda t: objective(t), n_trials=30, show_progress_bar=True)

    study_warm = crear_estudio_cluster(
        study, cluster_id=99,
        X_cluster=X_demo, y_cluster=y_demo,
        n_trials_warm=10, n_trials_new=20,
    )

    def best_curve(s):
        vals = [t.value for t in s.trials
                if t.state == optuna.trial.TrialState.COMPLETE]
        return np.minimum.accumulate(vals).tolist()

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=best_curve(study_cold),
                             name='cold start', line=dict(color='tomato')))
    fig.add_trace(go.Scatter(y=best_curve(study_warm),
                             name='warm start', line=dict(color='steelblue')))
    fig.update_layout(
        title='Cold start vs Warm start — convergencia RMSE',
        xaxis_title='trial', yaxis_title='RMSE', height=400,
    )
    fig.show()